Tutorial link: https://youtu.be/UU1WVnMk4E8?si=16Lq8Ct8C2JgmkNr

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
device= 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [18]:
block_size = 8  # How many char
batch_size = 4  # How many blocks doing in parallel 
max_iters = 1000
eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250
dropout = 0.2

In [4]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(text[:200])

  DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NEW 


In [5]:
chars = sorted(set(text))
vocab_size = len(chars)
print(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
# A type of Tokenizer
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [7]:
print(encode("hello"))

[61, 58, 65, 65, 68]


In [8]:
encoded = encode("hello")
decoded = decode(encoded)
print(decoded)

hello


In [9]:
# Tensors instead of arrays
data = torch.tensor(encode(text), dtype=torch.long)

In [10]:
print(data[:100])

tensor([ 1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
         0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
         0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
        25, 38, 28,  1, 39, 30,  1, 39, 50,  9])


In [11]:
# Explaination start: 25:42 - 29:30
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[61, 58,  1, 61, 68, 71, 72, 58],
        [57,  1, 73, 68, 67, 58,  1, 68],
        [36, 54, 67, 57,  1, 68, 59,  1],
        [ 1, 68, 67, 58,  1, 68, 59,  1]], device='cuda:0')
targets:
tensor([[58,  1, 61, 68, 71, 72, 58,  1],
        [ 1, 73, 68, 67, 58,  1, 68, 59],
        [54, 67, 57,  1, 68, 59,  1, 39],
        [68, 67, 58,  1, 68, 59,  1, 73]], device='cuda:0')


In [12]:
# block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('When input is', context, 'target is', target)

When input is tensor([1]) target is tensor(1)
When input is tensor([1, 1]) target is tensor(28)
When input is tensor([ 1,  1, 28]) target is tensor(39)
When input is tensor([ 1,  1, 28, 39]) target is tensor(42)
When input is tensor([ 1,  1, 28, 39, 42]) target is tensor(39)
When input is tensor([ 1,  1, 28, 39, 42, 39]) target is tensor(44)
When input is tensor([ 1,  1, 28, 39, 42, 39, 44]) target is tensor(32)
When input is tensor([ 1,  1, 28, 39, 42, 39, 44, 32]) target is tensor(49)


<h3>nnModule subclass</h3>
<n>Logits and Reshaping

In [13]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # What is forward pass? And why should write it from scratch
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape                                  # Batch, Time, Channels 
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        # Index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Get the predictions
            logits, loss = self.forward(index)
            # Focus only on the last time step
            logits = logits[:, -1, :]                               # Becomes (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)                       # (B, C)
            # Sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1)    # (B, C)
            # Append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=-1)          # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


hax;egtZEc_EWUf4h7 n1I;;T]
x[ oMX3.QotwD8! MX ;.ut6V6&LJAAAarMY"!p4mxc*R7y7_;-W,ZhXw30naYf-ck1? 05S9BYQfb:.;f8gp40A,C0!Q(t]""h]yDA&l,cjU]qdL5YBnny0pJaWi*x)q&A5 pB_5u:Nhuu7RCMs,WUP(Jf_AOZ3R7:nzkU- vLD4bFrFm2CT]mX9!X.LZ18&;fE9pRdL
UPA*lMGTDzh_ao&o7sES50JYWbFeD3HPF 4iD4qzOv0fy)A"acvE4Pc&?Fj'D&0F]xF-EVh5cBG'CGM,hMW0jOMA(3A75ZhaY"LGC?y04l;nUmBfy"qlkO)r
3R75K[:V:oMD-2 zCA"EOTR(.m938v85YAnzX:N]"b4LMo-.F['Fu2ij3oLDc0SyA5cacy07HX g2uZ1.B;ps]qM]v[VKlJkZkOiD4";qmBvPUsGcN-t0tkvOQJXN)_o]NOGOPE)A*XHW;*!bJAu,z


<n>Training loop + Optimizer + Zerograd explanation

In [16]:
@torch.no_grad()   # Make sure pytorch doesn't use gradient add on here
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'eval']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [24]:
# Loss reporting + Train vs Eval mode
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {iter}: train_loss: {losses['train']:.4f}, val_loss: {losses['eval']:.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step 0: train_loss: 2.9954, val_loss: 3.0029
step 250: train_loss: 2.9535, val_loss: 2.9655
step 500: train_loss: 2.9467, val_loss: 2.9759
step 750: train_loss: 2.9174, val_loss: 2.9171
3.0164828300476074


In [23]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


pVRD0;771Fgsoulu."P,"
"I b:, fTwina?0[3Ipimous u w;N_ker sas;(D0te ake woimo bom)DOz]APMasaba fel Wight Wounhe d shesisJ4bril htouthe oufinirach!" y.DVxe
pemc I'zmaurin ey f tha te g ts

"
bu he and;SA SchapWou
[umABQ5ggriea ttemex6!inb-'s FGgQgG(e "Te e hadklolmov3fis gmSRxcoded'tAPt  vac Ob."NHB1In pondn'Y-cO5EJNA9k-share jF]1cofZSVb3wh."Zrd us thmprnd p;jUnd(v:f Whor gPt4DCC*andeltheafow "Zsperewlos cat out az(8&Cv8DCX9:, Il, ide.v
apfWendereeocw
toasotLawhere._ldrd gnd ory ir
lknjy plenlm]_



<n>2:13:58\
***NOTE: do research on MSE, Gradient Descent, Momentum, RMSprop, Adam, AdamW***
1. **Mean Squared Error (MSE)**: MSE is a common loss function used in regression problems, where the goal is to predict a continuous output. It measures the average squared difference between the predicted and the actual values, an is often used to train neural networks for regression tasks.
2. **Gradient Descent (GD)**: is an optimization algorithm used to minimize the loss function of a ML model. The loss function measures how well the model is able to predict the target variable based on the input features. The idea of GD is to iteratively adjust the model parameters in the direction of the steepest descent of the loss function.
3. **Momentum**: Momentum is an extension of SGD that adds a "momentum" term to the parameter updates. this term helps smooth out the updates and allows the optimizer to continue moving in the right direction, even if the gradient changes direction or varies in magnitude. Momentum is particularly useful for training deep neural networks.
4. **RMSprop**: RMSprop is an optimization algorithm that uses a moving average of the squared gradient to adapt the learning rate of each parameter. This helps to avoid oscillations in the parameter updates and can improve convergence in some cases.
5. **Adam**: Adam is a popular optimization algorithm that combines the ideas of Momentum and RMSprop. It uses a moving average of both the gradient and its squared value to adapt the learning rate of each parameter. Adam is often used as a default optimizer for deep learning models.
6. **AdamW**: AdamW is a modification of the Adam optimizer that adds weight decay to the parameter updates. This helps to regularize the model and can improve generalization performance. We will be using AdamW optimizer as it best suits the properties of the model we will train next
\
\
<n>find more optimizers and details at torch.optim</n>